# BƯỚC 5.1: SINH DỮ LIỆU TƯƠNG TÁC MÔ PHỎNG
---
Lọc cộng tác (CF) cần **lịch sử tương tác của nhiều người dùng**, nhưng ứng dụng mới chạy nên log thật gần như trống (xem `data/interactions/`). Notebook này **sinh một bộ dữ liệu tương tác mô phỏng** trên **chính kho tin thật** để thiết kế, chọn tham số và đánh giá CF:

1. **Nạp kho tin IT** và gán nhóm nghề, vòng đời từng tin;
2. **Sinh dữ liệu**: thuộc tính ẩn của công ty / tin, 3.000 người dùng, mức thích thật $U$, các phiên hiển thị và phễu hành vi (xem ➔ lưu ➔ ứng tuyển);
3. **Lưu** vào `data/simulated/`.

**Tiếp theo:** notebook **02** xem dữ liệu vừa sinh trông như thế nào, rồi notebook **03** mới huấn luyện và đánh giá CF.

## 1. Nạp dữ liệu tin tuyển dụng IT

In [1]:
import os, re, json, time, unicodedata
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize as l2_normalize

pd.set_option('display.width', 160)
pd.set_option('display.max_columns', 20)

DATA_PATH = os.path.abspath(os.path.join('..', '..', 'data', 'notebook', 'content_base', 'jobs_clean.csv'))
OUT_DIR = os.path.abspath(os.path.join('..', '..', 'data', 'simulated'))
SEED = 42

jobs_all = pd.read_csv(DATA_PATH)
missing = {'is_it', 'required_skills', 'deadline'} - set(jobs_all.columns)
if missing:
    raise ValueError(f"jobs_clean.csv thiếu cột {missing}: hãy chạy lại notebook 02 trước.")
jobs = jobs_all[jobs_all['is_it'] == True].reset_index(drop=True)
jobs['skills'] = jobs['required_skills'].fillna('').map(lambda s: [x.strip() for x in s.split(',') if x.strip()])
print(f"Đã nạp {len(jobs):,} tin IT / {len(jobs_all):,} tin sạch, {jobs['company_name'].nunique():,} công ty")

Đã nạp 3,675 tin IT / 5,234 tin sạch, 1,712 công ty


### 1.1. Nhóm nghề và vòng đời của từng tin
- **Nhóm nghề**: suy từ tiêu đề bằng từ khóa; mỗi tin lấy nhóm khớp **đầu tiên** theo thứ tự ưu tiên trong bảng dưới (nhóm cụ thể đứng trước nhóm chung chung, vd. *Game Tester* vào Game chứ không vào QA). Mỗi người dùng mô phỏng thuộc đúng một nhóm và chủ yếu tìm việc trong nhóm của mình.

| Thứ tự | Nhóm nghề | Ví dụ tiêu đề |
|---|---|---|
| 1 | Game/Đồ họa | Game Designer, Unity Developer, 2D Animator, 3D Artist, Game Tester |
| 2 | QA/QC | Tester, QA Engineer, Kiểm thử phần mềm |
| 3 | Data/AI | Data Analyst, Data Engineer, AI Engineer, BI |
| 4 | BA/Product/PM | Business Analyst, Product Owner, Project Manager, BrSE, Scrum Master |
| 5 | UI/UX | UI/UX Designer, UX Research |
| 6 | Embedded/IoT | Kỹ sư nhúng, Firmware, PLC, Phần cứng |
| 7 | Frontend/Mobile | Frontend, React, Android, iOS, Flutter |
| 8 | DevOps/Cloud | DevOps, Cloud, Quản trị hệ thống, Mạng, An ninh mạng |
| 9 | Backend | Backend, Java, .NET, Golang, Fullstack, Lập trình viên |
| 10 | ERP/IT Support | SAP, ERP, IT Helpdesk, IT Support, Triển khai phần mềm |
| – | Khác | không khớp nhóm nào, **hoặc là vị trí kinh doanh** (*Sales phần mềm, Business Development, Presales*) |

  Từ khóa của từng nhóm nằm trong **`configs/dictionaries/job_domains.yaml`** 

In [2]:
import sys
ROOT = os.path.abspath(os.path.join('..', '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
# Nhóm nghề theo tiêu đề: từ khóa ở configs/dictionaries/job_domains.yaml (dùng chung với notebook 02)
from src.data_cleaning.job_domain import DOMAINS, DOMAIN_PATTERNS, job_domain, strip_accents

jobs['domain'] = jobs['title'].map(job_domain)

def infer_snapshot(df):
    """Ngày notebook 02 tính is_active: ngay sau hạn nộp muộn nhất của tin đã hết hạn (hoặc hạn sớm nhất của tin còn hạn)."""
    dl = pd.to_datetime(df['deadline'], errors='coerce')
    expired, active = dl[~df['is_active'].astype(bool)].dropna(), dl[df['is_active'].astype(bool)].dropna()
    if len(expired):
        return expired.max() + pd.Timedelta(days=1)
    return active.min() if len(active) else pd.Timestamp.now().normalize()

SNAPSHOT = infer_snapshot(jobs)
N_DAYS, JOB_OPEN_DAYS = 55, 45
DAY0 = SNAPSHOT - pd.Timedelta(days=N_DAYS)
deadline_day = (pd.to_datetime(jobs['deadline'], errors='coerce') - DAY0).dt.days
n_before = len(jobs)
jobs = jobs[deadline_day.notna() & (deadline_day >= 0)].reset_index(drop=True)      # bỏ tin đã đóng trước giai đoạn mô phỏng
deadline_day = deadline_day[deadline_day.notna() & (deadline_day >= 0)].values.astype(int)
jobs['open_day'] = np.clip(deadline_day - JOB_OPEN_DAYS, 0, N_DAYS)
jobs['close_day'] = np.clip(deadline_day, 0, N_DAYS)

print(f"Ngày chụp dữ liệu (suy từ file): {SNAPSHOT.date()} | giữ {len(jobs):,}/{n_before:,} tin có mở trong giai đoạn mô phỏng")
print("Phân bố nhóm nghề:", jobs['domain'].value_counts().to_dict())
print(f"Giai đoạn mô phỏng: {DAY0.date()} ➔ {SNAPSHOT.date()} ({N_DAYS} ngày)")
print(f"Số tin đang mở trung bình mỗi ngày: {np.mean([((jobs.open_day <= d) & (jobs.close_day >= d)).sum() for d in range(N_DAYS)]):,.0f}")

Ngày chụp dữ liệu (suy từ file): 2026-11-16 | giữ 3,675/3,675 tin có mở trong giai đoạn mô phỏng
Phân bố nhóm nghề: {'Khác': 641, 'DevOps/Cloud': 596, 'Data/AI': 519, 'BA/Product/PM': 408, 'Backend': 401, 'ERP/IT Support': 332, 'QA/QC': 280, 'Game/Đồ họa': 196, 'Frontend/Mobile': 132, 'Embedded/IoT': 101, 'UI/UX': 69}
Giai đoạn mô phỏng: 2026-09-22 ➔ 2026-11-16 (55 ngày)
Số tin đang mở trung bình mỗi ngày: 2,534


## 2. Sinh bộ dữ liệu mô phỏng

### 2.1. Thuộc tính ẩn của tin & công ty (mô hình KHÔNG được thấy)
Đây là thứ có thật trên thị trường nhưng **không nằm trong nội dung tin**, nên CBF không thể biết; chỉ hành vi của nhiều người mới để lộ ra:
- **Loại công ty** (4 loại: Startup, Product, Outsourcing, Tài chính/Doanh nghiệp lớn): gán ngẫu nhiên cho mỗi công ty.
- **Sức hút thương hiệu** của công ty: $b_c \sim \mathcal{N}(0, 1)$.
- **Độ hấp dẫn riêng** của tin (phúc lợi, dự án, đánh giá...): $q_j \sim \mathcal{N}(0, 0{,}5^2)$.

In [3]:
rng = np.random.default_rng(SEED)
COMPANY_TYPES = ['Startup', 'Product', 'Outsourcing', 'Tài chính / DN lớn']
companies = jobs['company_name'].fillna('?').unique()
company_type = dict(zip(companies, rng.choice(len(COMPANY_TYPES), size=len(companies), p=[0.2, 0.25, 0.35, 0.2])))
company_brand = dict(zip(companies, rng.normal(0, 1, len(companies))))
jobs['company_type'] = jobs['company_name'].fillna('?').map(company_type)
jobs['brand'] = jobs['company_name'].fillna('?').map(company_brand)
jobs['quality'] = rng.normal(0, 0.5, len(jobs))
print(jobs['company_type'].map(dict(enumerate(COMPANY_TYPES))).value_counts().to_string())

company_type
Outsourcing           1304
Product                815
Startup                789
Tài chính / DN lớn     767


### 2.2. Người dùng mô phỏng
Mỗi người dùng có **phần quan sát được** (thứ họ khai trong hồ sơ, CBF dùng được) và **phần ẩn** (khẩu vị, chỉ lộ qua hành vi):

| Thuộc tính | Cách sinh |
|---|---|
| Nhóm nghề | theo tỷ lệ tin của từng nhóm trong kho |
| Kỹ năng (3-7) | lấy mẫu từ **kỹ năng thật** của các tin cùng nhóm, kỹ năng phổ biến được chọn nhiều hơn |
| Kinh nghiệm | lấy mẫu từ yêu cầu kinh nghiệm của tin cùng nhóm, cộng thêm 0-2 năm |
| Thành phố | Hà Nội / Hồ Chí Minh / Đà Nẵng theo tỷ lệ tin, 10% chọn "Tất cả" |
| Mức hoạt động | số phiên ~ Poisson với trung bình lấy từ phân bố log-normal (ít người rất tích cực, nhiều người ít hoạt động) |
| **Khẩu vị loại công ty** *(ẩn)* | người dùng thuộc 1 trong 6 **cụm khẩu vị**; mỗi cụm thích các loại công ty khác nhau. Cấu trúc cụm này là thứ CF có thể khai thác |
| **Độ nhạy thương hiệu** *(ẩn)* | $\sim U(0, 1)$ |

In [4]:
N_USERS = 3000
N_TASTE_CLUSTERS = 6
skill_freq = {d: pd.Series([s for sk in jobs.loc[jobs.domain == d, 'skills'] for s in sk]).value_counts() for d in DOMAINS}
dom_share = jobs.loc[jobs.domain != 'Khác', 'domain'].value_counts(normalize=True)
city_share = jobs.loc[jobs.city.isin(['Hà Nội', 'Hồ Chí Minh', 'Đà Nẵng']), 'city'].value_counts(normalize=True)
cluster_taste = rng.dirichlet([0.4] * len(COMPANY_TYPES), size=N_TASTE_CLUSTERS)    # mỗi cụm: phân bố khẩu vị 4 loại công ty

users = []
for u in range(N_USERS):
    d = rng.choice(dom_share.index, p=dom_share.values)
    freq = skill_freq[d]
    k = int(rng.integers(3, 8))
    p = freq.values ** 0.7; p = p / p.sum()
    skills = list(rng.choice(freq.index, size=min(k, len(freq)), replace=False, p=p))
    exp_pool = jobs.loc[jobs.domain == d, 'experience_min'].dropna().values
    cl = int(rng.integers(N_TASTE_CLUSTERS))
    taste = cluster_taste[cl] * 0.8 + rng.dirichlet([1] * len(COMPANY_TYPES)) * 0.2
    users.append({
        'user_id': u, 'domain': d, 'skills': skills,
        'exp': float(np.clip(rng.choice(exp_pool) + rng.uniform(0, 2), 0, 15)),
        'city': 'Tất cả' if rng.random() < 0.10 else rng.choice(city_share.index, p=city_share.values),
        'activity': float(rng.lognormal(np.log(5), 0.8)),
        'taste_cluster': cl, 'taste': taste, 'brand_sens': float(rng.uniform(0, 1)),
    })
users = pd.DataFrame(users)
print(f"{len(users):,} người dùng mô phỏng")
print("Theo nhóm nghề:", users['domain'].value_counts().to_dict())
display(users[['user_id', 'domain', 'skills', 'exp', 'city', 'activity', 'taste_cluster']].head(5))

3,000 người dùng mô phỏng
Theo nhóm nghề: {'DevOps/Cloud': 591, 'Data/AI': 509, 'BA/Product/PM': 412, 'Backend': 375, 'ERP/IT Support': 364, 'QA/QC': 272, 'Game/Đồ họa': 193, 'Frontend/Mobile': 114, 'Embedded/IoT': 101, 'UI/UX': 69}


,user_id,domain,skills,exp,city,activity,taste_cluster
0,0,Data/AI,"[Power BI, MongoDB, Terraform]",1.319385,Hà Nội,3.655424,4
1,1,DevOps/Cloud,"[Node.js, AWS, Bash, System Administration, An...",3.917168,Hà Nội,10.346902,4
2,2,Frontend/Mobile,"[Docker, Postman, Angular, Java, CI/CD, Photos...",2.264286,Hà Nội,6.550750,4
3,3,DevOps/Cloud,"[VMware, SQL, Git, Jenkins, Docker]",1.977920,Hồ Chí Minh,5.304076,1
4,4,ERP/IT Support,"[PostgreSQL, Pandas, LLM, SQL, An Ninh Mạng]",3.680683,Hồ Chí Minh,3.167592,2


### 2.3. Mức độ thích thật $U(u, j)$ (ground truth)

$$U_\alpha(u, j) = \alpha \cdot z\big(\text{Nội dung}(u, j)\big) + (1 - \alpha) \cdot z\big(\text{Ẩn}(u, j)\big)$$

- **Nội dung** (CBF quan sát được): $0{,}45 \cdot$ khớp kỹ năng $+ 0{,}25 \cdot$ đúng nhóm nghề $+ 0{,}15 \cdot$ đủ kinh nghiệm $+ 0{,}15 \cdot$ đúng thành phố. Khớp kỹ năng $= \frac{|S_u \cap S_j|}{\sqrt{|S_u| \cdot |S_j|}}$.
- **Ẩn**: $\text{khẩu vị}_u[\text{loại công ty}_j] \cdot 3 + \text{độ nhạy thương hiệu}_u \cdot b_c + q_j$.
- $z(\cdot)$: chuẩn hóa về trung bình 0, độ lệch chuẩn 1 để hai phần có cùng thang đo; $\alpha$ là **núm vặn** của thí nghiệm.

Phần ẩn được sinh **từ cụm khẩu vị + thương hiệu**, *không phải* tích hai vector ngẫu nhiên. Nếu sinh đúng dạng "tích vector", Matrix Factorization sẽ khớp hoàn hảo với cơ chế sinh và thắng một cách dễ dãi.

In [5]:
skill_vocab = {s: i for i, s in enumerate(sorted({s for sk in jobs['skills'] for s in sk}))}

def skill_matrix(skill_lists):
    rows, cols = [], []
    for r, sk in enumerate(skill_lists):
        for s in sk:
            if s in skill_vocab:
                rows.append(r); cols.append(skill_vocab[s])
    return sp.csr_matrix((np.ones(len(rows), dtype=np.float32), (rows, cols)), shape=(len(skill_lists), len(skill_vocab)))

SU, SJ = skill_matrix(users['skills']), skill_matrix(jobs['skills'])
overlap = (SU @ SJ.T).toarray()
denom = np.sqrt(np.outer(np.asarray(SU.sum(1)).ravel(), np.asarray(SJ.sum(1)).ravel()))
skill_fit = np.divide(overlap, denom, out=np.zeros_like(overlap), where=denom > 0)

domain_fit = (users['domain'].to_numpy(dtype=object)[:, None] == jobs['domain'].to_numpy(dtype=object)[None, :]).astype(np.float32)
job_exp = jobs['experience_min'].fillna(0).values
exp_fit = np.clip(1 - np.maximum(job_exp[None, :] - users['exp'].values[:, None], 0) / 3, 0, 1)
ucity, jcity = users['city'].to_numpy(dtype=object)[:, None], jobs['city'].to_numpy(dtype=object)[None, :]
city_fit = np.where((ucity == 'Tất cả') | (ucity == jcity) | (jcity == 'Toàn quốc'), 1.0, 0.2)
content = 0.45 * skill_fit + 0.25 * domain_fit + 0.15 * exp_fit + 0.15 * city_fit

taste = np.stack(users['taste'].values)                                   # (n_users, 4)
hidden = (taste[:, jobs['company_type'].values] * 3
          + users['brand_sens'].values[:, None] * jobs['brand'].values[None, :]
          + jobs['quality'].values[None, :])

zscore = lambda M: ((M - M.mean()) / M.std()).astype(np.float32)
Z_CONTENT, Z_HIDDEN = zscore(content), zscore(hidden)
del overlap, denom, skill_fit, domain_fit, exp_fit, city_fit, content, hidden

def true_utility(alpha):
    return alpha * Z_CONTENT + (1 - alpha) * Z_HIDDEN

print("Ma trận U(u, j):", Z_CONTENT.shape, "| tương quan Nội dung ↔ Ẩn:",
      round(float(np.corrcoef(Z_CONTENT.ravel()[::97], Z_HIDDEN.ravel()[::97])[0, 1]), 3))

Ma trận U(u, j): (3000, 3675) | tương quan Nội dung ↔ Ẩn: -0.005


### 2.4. Mô phỏng phiên: người dùng chỉ thấy một phần nhỏ kho tin

Mỗi phiên (một ngày ngẫu nhiên), người dùng thấy **20 tin đang mở**:

| Nguồn | Số tin | Ý nghĩa |
|---|---|---|
| Tìm kiếm | 10 | tin cùng nhóm nghề, chọn ngẫu nhiên (mô phỏng gõ từ khóa) |
| Gợi ý của hệ thống | 6 | lấy từ top-30 theo **điểm nội dung có nhiễu** (mô phỏng một hệ CBF đang chạy) |
| Khám phá | 4 | ngẫu nhiên trong toàn kho đang mở |

Thứ tự hiển thị được xáo trộn, rồi áp dụng **phễu hành vi**:
- Xem tới vị trí $k$ với xác suất $1/\log_2(k + 2)$ (*thiên lệch vị trí*: tin ở trên được chú ý nhiều hơn);
- **Bấm xem** $\sim \sigma(2U - 4)$; nếu đã bấm: **lưu** $\sim \sigma(U - 3)$, **ứng tuyển** $\sim \sigma(U - 4)$.

Các hằng số được hiệu chỉnh để tỷ lệ bấm khoảng 5-6% số lượt hiển thị, và trong số lượt bấm thì phần lớn chỉ xem, ít người lưu, ứng tuyển hiếm nhất (xem mục 2.5).

Bước hiển thị **không phụ thuộc $\alpha$**, và các số ngẫu nhiên của phễu được sinh **một lần** rồi dùng chung cho mọi $\alpha$ (*common random numbers*). Nhờ vậy khác biệt giữa các mức $\alpha$ chỉ đến từ $\alpha$, không phải từ may rủi.

In [6]:
SLATE = {'search': 10, 'feed': 6, 'explore': 4}
open_d, close_d = jobs['open_day'].values, jobs['close_day'].values
job_dom = jobs['domain'].to_numpy(dtype=object)

t0 = time.time()
imp = []   # (user, job, day, position, source)
for u in users.itertuples():
    n_sessions = max(1, rng.poisson(u.activity))
    for day in np.sort(rng.integers(0, N_DAYS, size=n_sessions)):
        alive = np.flatnonzero((open_d <= day) & (close_d >= day))
        same = alive[job_dom[alive] == u.domain]
        picks = []
        if len(same):
            picks += [(j, 'search') for j in rng.choice(same, size=min(SLATE['search'], len(same)), replace=False)]
        noisy = Z_CONTENT[u.user_id, alive] + rng.normal(0, 0.3, len(alive))
        top = alive[np.argsort(-noisy)[:30]]
        picks += [(j, 'feed') for j in rng.choice(top, size=SLATE['feed'], replace=False)]
        picks += [(j, 'explore') for j in rng.choice(alive, size=SLATE['explore'], replace=False)]
        seen, slate = set(), []
        for j, src in picks:                       # bỏ tin trùng trong cùng một phiên
            if j not in seen:
                seen.add(j); slate.append((j, src))
        order = rng.permutation(len(slate))
        for pos, k in enumerate(order):
            imp.append((u.user_id, slate[k][0], day, pos, slate[k][1]))

impressions = pd.DataFrame(imp, columns=['user_id', 'job_idx', 'day', 'position', 'source'])
n = len(impressions)
impressions['r_examine'], impressions['r_click'], impressions['r_save'], impressions['r_apply'] = rng.random((4, n)).astype(np.float32)
print(f"{n:,} lượt hiển thị trong {impressions.groupby(['user_id', 'day']).ngroups:,} phiên ({time.time() - t0:.1f}s)")

403,653 lượt hiển thị trong 18,450 phiên (6.2s)


In [7]:
sigmoid = lambda x: 1 / (1 + np.exp(-x))
EVENT_WEIGHT = {'click': 1.0, 'save': 3.0, 'apply': 5.0}
FUNNEL = {'click': (2.0, -4.0), 'save': (1.0, -3.0), 'apply': (1.0, -4.0)}   # (hệ số của U, hằng số) trong σ(a·U + b)

def simulate_events(alpha):
    """Áp dụng phễu hành vi lên các lượt hiển thị ➔ bảng sự kiện (mỗi lượt hiển thị tối đa 1 sự kiện: hành động sâu nhất)."""
    Ua = true_utility(alpha)[impressions['user_id'].values, impressions['job_idx'].values]
    examined = impressions['r_examine'].values < 1 / np.log2(impressions['position'].values + 2)
    p = {e: sigmoid(a * Ua + b) for e, (a, b) in FUNNEL.items()}
    click = examined & (impressions['r_click'].values < p['click'])
    save = click & (impressions['r_save'].values < p['save'])
    apply_ = click & (impressions['r_apply'].values < p['apply'])
    ev = impressions.loc[click, ['user_id', 'job_idx', 'day', 'position', 'source']].copy()
    ev['event'] = np.where(apply_[click], 'apply', np.where(save[click], 'save', 'click'))
    ev['weight'] = ev['event'].map(EVENT_WEIGHT)
    return ev.reset_index(drop=True)

events_05 = simulate_events(0.5)
print(f"α = 0,5: {len(events_05):,} sự kiện | " + ", ".join(f"{k}: {v:,}" for k, v in events_05['event'].value_counts().items()))

α = 0,5: 27,931 sự kiện | click: 19,272, save: 5,674, apply: 2,985


## 3. Lưu bộ dữ liệu mô phỏng
Lưu vào `data/simulated/` (thư mục `data/` không đưa lên git; chạy lại notebook sẽ sinh lại y hệt nhờ cố định `SEED`).

| File | Nội dung |
|---|---|
| `users.csv` | hồ sơ người dùng; các cột `taste_cluster`, `taste`, `brand_sens` là **phần ẩn**, chỉ dùng để phân tích |
| `jobs_hidden.csv` | `job_id`, nhóm nghề, vòng đời, loại công ty, thương hiệu, độ hấp dẫn (**ẩn**) |
| `impressions.csv` | mọi lượt hiển thị: người dùng, tin, ngày, vị trí, nguồn |
| `interactions_alpha0.5.csv` | sự kiện bấm / lưu / ứng tuyển ở $\alpha = 0{,}5$: **thứ CF được học** |
| `true_utility_alpha0.5.npy` | ma trận $U(u, j)$ thật ở $\alpha = 0{,}5$, **chỉ dùng để chấm điểm** (notebook 07 và ứng dụng đọc file này) |
| `z_content.npy`, `z_hidden.npy` | hai thành phần của $U$: notebook 03 ghép lại thành $U_\alpha$ cho mọi $\alpha$ |
| `impression_draws.npy` | các số ngẫu nhiên của phễu hành vi: notebook 03 sinh lại sự kiện ở mức $\alpha$ khác (*common random numbers*) |
| `config.json` | toàn bộ tham số mô phỏng; notebook 03 ghi thêm tham số ALS và trọng số Hybrid đã chọn |

In [8]:
os.makedirs(OUT_DIR, exist_ok=True)
u_out = users.copy()
u_out['skills'] = u_out['skills'].map(', '.join)
u_out['taste'] = u_out['taste'].map(lambda t: json.dumps([round(float(x), 4) for x in t]))
u_out.to_csv(os.path.join(OUT_DIR, 'users.csv'), index=False, encoding='utf-8')
jh = jobs[['job_id', 'domain', 'open_day', 'close_day', 'company_type', 'brand', 'quality']].copy()
jh['company_type'] = jh['company_type'].map(dict(enumerate(COMPANY_TYPES)))
jh.to_csv(os.path.join(OUT_DIR, 'jobs_hidden.csv'), index=False, encoding='utf-8')
job_ids = jobs['job_id'].to_numpy(dtype=object)
impressions.assign(job_id=job_ids[impressions.job_idx])[['user_id', 'job_id', 'day', 'position', 'source']] \
    .to_csv(os.path.join(OUT_DIR, 'impressions.csv'), index=False)
events_05.assign(job_id=job_ids[events_05.job_idx])[['user_id', 'job_id', 'day', 'position', 'source', 'event', 'weight']] \
    .to_csv(os.path.join(OUT_DIR, 'interactions_alpha0.5.csv'), index=False)
np.save(os.path.join(OUT_DIR, 'true_utility_alpha0.5.npy'), true_utility(0.5).astype(np.float16))
# Để notebook 03 tái tạo ĐÚNG sở thích thật ở mọi α và sinh lại sự kiện cho phân tích độ nhạy:
np.save(os.path.join(OUT_DIR, 'z_content.npy'), Z_CONTENT)                                  # z(Nội dung), float32
np.save(os.path.join(OUT_DIR, 'z_hidden.npy'), Z_HIDDEN)                                    # z(Ẩn), float32
np.save(os.path.join(OUT_DIR, 'impression_draws.npy'),                                      # số ngẫu nhiên của phễu, cùng thứ tự với impressions.csv
        impressions[['r_examine', 'r_click', 'r_save', 'r_apply']].to_numpy(np.float32))

# Tham số ALS và trọng số Hybrid do notebook 03 chọn và ghi thêm vào config.json: giữ lại giá trị cũ nếu đã có
old_cfg = {}
if os.path.exists(os.path.join(OUT_DIR, 'config.json')):
    old_cfg = json.load(open(os.path.join(OUT_DIR, 'config.json'), encoding='utf-8'))
config = {'seed': SEED, 'n_users': N_USERS, 'n_days': N_DAYS, 'job_open_days': JOB_OPEN_DAYS, 'snapshot': str(SNAPSHOT.date()),
          'taste_clusters': N_TASTE_CLUSTERS, 'company_types': COMPANY_TYPES, 'slate': SLATE, 'event_weight': EVENT_WEIGHT,
          'funnel': FUNNEL, 'val_start_day': 38, 'test_start_day': 45,
          'snapshot_day0': str(DAY0.date()), 'utility': 'alpha*z(content) + (1-alpha)*z(hidden)',
          **{k: old_cfg[k] for k in ('als_params', 'w_cbf') if k in old_cfg}}
with open(os.path.join(OUT_DIR, 'config.json'), 'w', encoding='utf-8') as f:
    json.dump(config, f, ensure_ascii=False, indent=2)
for fn in sorted(os.listdir(OUT_DIR)):
    print(f"  {fn:32s} {os.path.getsize(os.path.join(OUT_DIR, fn)) / 1e6:6.1f} MB")

  config.json                         0.0 MB
  impression_draws.npy                6.5 MB
  impressions.csv                    13.6 MB
  interactions_alpha0.5.csv           1.2 MB
  jobs_hidden.csv                     0.3 MB
  true_utility_alpha0.5.npy          22.1 MB
  users.csv                           0.5 MB
  z_content.npy                      44.1 MB
  z_hidden.npy                       44.1 MB
